## **집계연산**

### **PySpark 설정**

In [2]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext

In [3]:
# [+] SparkConf, SparkContext 객체 생성하기
conf = SparkConf().setMaster('local').setAppName('key-value-rdd')

In [4]:
sc = SparkContext(conf=conf)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 04:14:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **reduce( ) 메서드**

In [5]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x,y: x+y)

15

In [6]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()

[1, 2, 3, 4, 5]

In [7]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [8]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
sc.parallelize([1,2,3,4,5]).reduce(lambda x,y: (x*2) + y)

57

In [9]:
sc.parallelize([1,2,3,4,5],3).reduce(lambda x,y: (x*2) + y)

31

In [10]:
sc.parallelize([1,2,3,4,5],4).reduce(lambda x,y: (x*2) + y)

35

In [11]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5]).reduce(lambda x,y: (x*2) +y)

57

In [12]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

### **10000 개 정수 난수의 총합 계산하기**

In [13]:
!pip install numpy

In [14]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [15]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0, 100, size = 10000)
x[:100]

array([11, 90, 51, 54, 48,  7, 84, 95, 99, 90, 96, 33, 48, 87, 59, 96, 48,
       72, 53, 22,  4, 55, 36, 79, 62, 47,  7, 84, 28, 49,  6, 70, 65, 51,
       15, 40, 36, 94,  2, 40, 88, 71, 67, 11,  4, 48, 53, 54, 23, 58,  5,
       55, 74, 41, 96, 81, 67, 19, 42, 40, 74, 37, 95, 19, 61, 95, 53, 63,
       25, 75, 82, 35, 17, 65, 67, 42,  0, 19, 36, 50, 93,  3, 76, 79, 40,
       99,  7,  0, 79, 39, 14, 42, 17,  5, 27, 47, 38, 60, 91, 94])

In [16]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x,8)

In [17]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

Partition 0: [np.int64(11), np.int64(90), np.int64(51), np.int64(54), np.int64(48), np.int64(7), np.int64(84), np.int64(95), np.int64(99), np.int64(90), np.int64(96), np.int64(33), np.int64(48), np.int64(87), np.int64(59), np.int64(96), np.int64(48), np.int64(72), np.int64(53), np.int64(22), np.int64(4), np.int64(55), np.int64(36), np.int64(79), np.int64(62), np.int64(47), np.int64(7), np.int64(84), np.int64(28), np.int64(49), np.int64(6), np.int64(70), np.int64(65), np.int64(51), np.int64(15), np.int64(40), np.int64(36), np.int64(94), np.int64(2), np.int64(40), np.int64(88), np.int64(71), np.int64(67), np.int64(11), np.int64(4), np.int64(48), np.int64(53), np.int64(54), np.int64(23), np.int64(58), np.int64(5), np.int64(55), np.int64(74), np.int64(41), np.int64(96), np.int64(81), np.int64(67), np.int64(19), np.int64(42), np.int64(40), np.int64(74), np.int64(37), np.int64(95), np.int64(19), np.int64(61), np.int64(95), np.int64(53), np.int64(63), np.int64(25), np.int64(75), np.int64(82),

In [18]:
# [+] x+y 집계 연산
rdd.reduce(lambda x,y: x+y)

np.int64(495655)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [19]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x,y : x+y)

12

In [20]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1,2,3,4],2).fold(1, lambda x,y : x+y)

13

In [21]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x,y : x*y)

0

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [23]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([3,10,7,5,12,8,10]).groupBy(lambda x : x%2).collect()

In [24]:
# [+] groupBy() 결과 출력하기
for k, v in res:
    print(k, list(v))

1 [3, 7, 5]
0 [10, 12, 8, 10]


### **aggregate( ) 메서드**

In [25]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4])\
    .reduce(lambda x, y: (x + y, x * y)) 
    # single value -> pair 생성 시도할 경우 오류

26/10/08 04:25:58 ERROR Executor: Exception in task 0.0 in stage 15.0 (TID 38)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 2044, in main
    process()
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 2036, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 273, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/core/rdd.py", line 1789, in func
    yield reduce(f, iterator, initial)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/util.py", line 131, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1228/3056019969.py", line 3, in <lambda>
TypeError: can only concatenate tuple (not "int") to

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 15.0 failed 1 times, most recent failure: Lost task 0.0 in stage 15.0 (TID 38) (0fd83eb7092a executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 2044, in main
    process()
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 2036, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 273, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/core/rdd.py", line 1789, in func
    yield reduce(f, iterator, initial)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/p

In [26]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

TypeError: unsupported operand type(s) for +: 'int' and 'tuple'

In [27]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [28]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)

(10, 4)